# Milestone 1 — Corpus Statistics & Analysis

> **Compliance Note**: All figures, distributions, and tables in this notebook are **dynamically generated at build time** from reproducible pipeline measurement data (`book/data/summary.json`). No static screenshot images are used.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

# Locate and load pipeline summary data
data_dir = Path("../../data")
summary_path = data_dir / "summary.json"
papers_path = data_dir / "papers_unique.csv"

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

df_sample = pd.read_csv(papers_path) if papers_path.exists() else pd.DataFrame()

## 1. Corpus Key Performance Indicators (KPIs)

The table below computes success metrics with explicit, audited denominators. As established in the technical plan, coverage considers valid `no_tex` publications as valid entries rather than omissions.

In [ ]:
kpi_metrics = [
    {"Dimension": "Paper Discovery", "Metric": "Total Assigned Papers", "Observed Value": f"{summary.get('papers_discovered', 0):,}", "Benchmark Target": "20,000 papers"},
    {"Dimension": "Paper Discovery", "Metric": "Full Source Papers (All Versions TeX)", "Observed Value": f"{summary.get('full_source_papers', 0):,}", "Benchmark Target": f"{summary.get('paper_success_rate', 0)*100:.2f}% rate"},
    {"Dimension": "Version Coverage", "Metric": "Total Expected Versions", "Observed Value": f"{summary.get('versions_expected', 0):,}", "Benchmark Target": "Full v1..vN history"},
    {"Dimension": "Version Coverage", "Metric": "Source Coverage (TeX + valid no_tex)", "Observed Value": f"{summary.get('source_coverage_rate', 0)*100:.2f}%", "Benchmark Target": "> 99.0%"},
    {"Dimension": "Reference Mining", "Metric": "Reference Jobs Success Rate", "Observed Value": f"{summary.get('reference_success_rate', 0)*100:.2f}%", "Benchmark Target": "Semantic Scholar API"},
    {"Dimension": "Reference Mining", "Metric": "Average References / Paper", "Observed Value": f"{summary.get('average_reference_count', 0):.2f}", "Benchmark Target": f"N = {summary.get('reference_count_denominator', 0):,}"},
    {"Dimension": "Reference Mining", "Metric": "Average Reference Completeness", "Observed Value": f"{summary.get('average_reference_completeness_rate', 0)*100:.2f}%", "Benchmark Target": "arXiv pool completeness"},
    {"Dimension": "Storage Footprint", "Metric": "Avg Version Size (Before Filter)", "Observed Value": f"{summary.get('average_source_bytes_before', 0)/1024:.2f} KB", "Benchmark Target": "Raw source archive"},
    {"Dimension": "Storage Footprint", "Metric": "Avg Version Size (After Filter)", "Observed Value": f"{summary.get('average_source_bytes_after', 0)/1024:.2f} KB", "Benchmark Target": ".tex/.bib whitelist only"}
]

df_kpi = pd.DataFrame(kpi_metrics)
display(df_kpi.style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2b303a'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f8f9fa')]}
]))

## 2. Storage Optimization via Figure Removal

Selective extraction removes heavy image binaries (PNG, JPG, PDF) while preserving 100% of the textual and mathematical structure. The interactive bar chart below illustrates the dramatic reduction in per-version storage size.

In [ ]:
bytes_before = summary.get('average_source_bytes_before', 0) / 1024
bytes_after = summary.get('average_source_bytes_after', 0) / 1024
saved_pct = ((bytes_before - bytes_after) / bytes_before * 100) if bytes_before > 0 else 0

fig_size = go.Figure(data=[
    go.Bar(
        name='Before Filtering (Full Archive)',
        x=['Source Size per Version'],
        y=[bytes_before],
        marker_color='#e74c3c',
        text=[f"{bytes_before:.1f} KB"],
        textposition='auto'
    ),
    go.Bar(
        name='After Filtering (.tex / .bib only)',
        x=['Source Size per Version'],
        y=[bytes_after],
        marker_color='#2ecc71',
        text=[f"{bytes_after:.1f} KB"],
        textposition='auto'
    )
])

fig_size.update_layout(
    title=f"<b>Average Source Size Optimization ({saved_pct:.1f}% Storage Saved)</b>",
    yaxis_title="Kilobytes (KB)",
    barmode='group',
    template='plotly_white',
    height=450,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig_size.show()

## 3. Reference Pool Breakdown

Our pipeline distinguishes between the full citations returned by Semantic Scholar and the clean subset of citations that resolve to valid `arxiv_id`s, forming the core matching pool for Milestone 3.

In [ ]:
total_refs = summary.get('crawled_reference_rows', 0)
arxiv_refs = summary.get('arxiv_reference_rows', 0)
non_arxiv_refs = summary.get('non_arxiv_reference_rows', 0)

fig_donut = go.Figure(data=[go.Pie(
    labels=['arXiv-keyed References (Matching Pool)', 'Non-arXiv References (Retained in Local Store)'],
    values=[arxiv_refs, non_arxiv_refs],
    hole=0.45,
    marker_colors=['#3498db', '#95a5a6']
)])

fig_donut.update_layout(
    title=f"<b>Reference Distribution: Total Crawled = {total_refs:,} Rows</b>",
    template='plotly_white',
    height=450
)
fig_donut.show()

## 4. Crawled Corpus Sample Preview

A sample preview of publications successfully crawled across the team partitions:

In [ ]:
if not df_sample.empty:
    preview_cols = ['member', 'arxiv_id', 'title', 'submission_date']
    display(df_sample[preview_cols].style.hide(axis="index").set_properties(**{'text-align': 'left'}))
else:
    print("No sample papers found.")